In [4]:
%pip install -q ultralytics

import cv2
import json
import random
import shutil
import zipfile
import yaml
import torch
from pathlib import Path
from PIL import Image, ImageOps
from tqdm import tqdm
from google.colab import drive
from ultralytics import YOLO

# 1. Drive Bağlantısı ve Dosya Kontrolleri
drive.mount("/content/drive")

zip_path = Path("/content/drive/MyDrive/training_data/quadrant-enumeration-disease.zip")
g2_zip_path = Path("/content/drive/MyDrive/training_data/gorev2_best.pt.zip")
g2_raw_path = Path("/content/drive/MyDrive/training_data/gorev2_best.pt")

if not zip_path.exists():
    raise FileNotFoundError(f"ZIP bulunamadı: {zip_path}")

if g2_zip_path.exists():
    g2_extract_dir = Path("/content/g2_weights")
    if g2_extract_dir.exists():
        shutil.rmtree(g2_extract_dir)
    g2_extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(g2_zip_path, "r") as zf:
        zf.extractall(g2_extract_dir)

    pt_files = [
        f for f in g2_extract_dir.rglob("*.pt*")
        if not f.name.startswith("._") and "__MACOSX" not in f.parts
    ]
    if pt_files:
        g2_weights_path = pt_files[0]
    elif list(g2_extract_dir.rglob("data.pkl")):
        g2_weights_path = Path("/content/gorev2_best.pt")
        shutil.copy(g2_zip_path, g2_weights_path)
    else:
        icerik = [str(p.relative_to(g2_extract_dir)) for p in g2_extract_dir.rglob("*") if p.is_file()]
        raise FileNotFoundError(f"Arşiv içeriği tanınamadı. Dosyalar: {icerik[:10]}")
elif g2_raw_path.exists():
    g2_weights_path = g2_raw_path
else:
    raise FileNotFoundError("Ne gorev2_best.pt.zip ne de gorev2_best.pt bulunamadı!")

print("Görev 2 ağırlık dosyası hazır:", g2_weights_path)

extract_root = Path("/content/dentex_q3_exp03")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_root)

coco_path = list(extract_root.rglob("train_quadrant_enumeration_disease.json"))[0]
BASE_DIR = coco_path.parent
images_src = BASE_DIR / "xrays"

image_files = sorted([f for f in images_src.glob("*.*") if f.suffix.lower() in [".png", ".jpg", ".jpeg"]])
assert len(image_files) == 705, f"Beklenen 705 görüntü, bulunan: {len(image_files)}"

# 2. Sena'nın Görev 2 Fonksiyonu (Tam Panoramik Diş Tespiti)
g2_model = YOLO(str(g2_weights_path))
OVERLAP_RATIO = 0.05

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def detect_all_teeth_sena(image_path: Path, conf_thres: float = 0.40, iou_thres: float = 0.50):
    pil_img = Image.open(image_path).convert("RGB")
    img_w, img_h = pil_img.size
    mid_x = img_w / 2.0
    overlap_px = img_w * OVERLAP_RATIO

    right_crop_x2 = int(min(img_w, mid_x + overlap_px))
    right_img = pil_img.crop((0, 0, right_crop_x2, img_h))

    left_crop_x1 = int(max(0, mid_x - overlap_px))
    left_w = img_w - left_crop_x1
    left_img = pil_img.crop((left_crop_x1, 0, img_w, img_h))
    left_img_flipped = ImageOps.mirror(left_img)

    res_r = g2_model.predict(right_img, conf=conf_thres, verbose=False)[0]
    res_l = g2_model.predict(left_img_flipped, conf=conf_thres, verbose=False)[0]

    raw_dets = []
    for box in res_r.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        raw_dets.append({"bbox": [float(x1), float(y1), float(x2), float(y2)], "conf": float(box.conf[0].item())})

    for box in res_l.boxes:
        fx1, y1, fx2, y2 = box.xyxy[0].cpu().numpy()
        orig_x1 = (left_w - fx2) + left_crop_x1
        orig_x2 = (left_w - fx1) + left_crop_x1
        raw_dets.append({"bbox": [float(orig_x1), float(y1), float(orig_x2), float(y2)], "conf": float(box.conf[0].item())})

    raw_dets.sort(key=lambda d: d["conf"], reverse=True)
    final_dets = []
    for det in raw_dets:
        if all(compute_iou(det["bbox"], kept) < iou_thres for kept in final_dets):
            final_dets.append(det["bbox"])
    return final_dets

# 3. 4 Anomali + 5. Sınıf (Healthy) Etiket Üretimi
with open(coco_path, "r") as f:
    data = json.load(f)

images_info = {
    img["id"]: (Path(img["file_name"]).stem, img["width"], img["height"])
    for img in data["images"]
}
stem_to_id = {Path(img["file_name"]).stem: img["id"] for img in data["images"]}

categories_3 = sorted(data["categories_3"], key=lambda x: x["id"])
cat_id_map = {cat["id"]: idx for idx, cat in enumerate(categories_3)}
class_names = {idx: cat["name"] for idx, cat in enumerate(categories_3)}
HEALTHY_ID = 4
class_names[HEALTHY_ID] = "Healthy"
print("5 Sınıflı Eğitim Etiketleri:", class_names)

anns_by_img = {img["id"]: [] for img in data["images"]}
for ann in data["annotations"]:
    anns_by_img[ann["image_id"]].append(ann)

labels_dir = BASE_DIR / "labels"
labels_dir.mkdir(parents=True, exist_ok=True)

healthy_added_total = 0
anomaly_total = 0

print("Tüm röntgenlerde sağlıklı dişler pseudo-labeling ile işaretleniyor...")
for img_path in tqdm(image_files):
    stem = img_path.stem
    img_id = stem_to_id[stem]
    _, img_w, img_h = images_info[img_id]

    lines = []
    disease_boxes_xyxy = []

    # A) Gerçek anomali kutuları
    for ann in anns_by_img[img_id]:
        cls_idx = cat_id_map[ann["category_id_3"]]
        x_min, y_min, w, h = ann["bbox"]
        disease_boxes_xyxy.append([x_min, y_min, x_min + w, y_min + h])

        xc = max(0.0, min(1.0, (x_min + w / 2.0) / img_w))
        yc = max(0.0, min(1.0, (y_min + h / 2.0) / img_h))
        wn = max(0.0, min(1.0, w / img_w))
        hn = max(0.0, min(1.0, h / img_h))
        lines.append(f"{cls_idx} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
        anomaly_total += 1

    # B) Görev 2 modeliyle bulunan ve anomalilerle çakışmayan sağlıklı dişler
    all_teeth = detect_all_teeth_sena(img_path, conf_thres=0.40)
    for t_box in all_teeth:
        if all(compute_iou(t_box, d_box) < 0.30 for d_box in disease_boxes_xyxy):
            tx1, ty1, tx2, ty2 = t_box
            tw, th = tx2 - tx1, ty2 - ty1
            xc = max(0.0, min(1.0, (tx1 + tw / 2.0) / img_w))
            yc = max(0.0, min(1.0, (ty1 + th / 2.0) / img_h))
            wn = max(0.0, min(1.0, tw / img_w))
            hn = max(0.0, min(1.0, th / img_h))
            lines.append(f"{HEALTHY_ID} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
            healthy_added_total += 1

    with open(labels_dir / f"{stem}.txt", "w") as f:
        f.write("\n".join(lines))

print(f"Etiketleme Bitti -> Anomali Kutusu: {anomaly_total} | Eklenen Sağlıklı Diş Kutusu: {healthy_added_total}")

# 4. Aynı Bölünme (seed=42) + CLAHE Ön İşleme
def apply_clahe(img_bgr):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return cv2.cvtColor(cv2.merge((clahe.apply(l), a, b)), cv2.COLOR_LAB2BGR)

random.seed(42)
random.shuffle(image_files)

n_total = len(image_files)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

splits = {
    "train": image_files[:n_train],
    "val": image_files[n_train:n_train + n_val],
    "test": image_files[n_train + n_val:]
}

for split_name, file_list in splits.items():
    img_dst = BASE_DIR / "images" / split_name
    lbl_dst = BASE_DIR / "labels" / split_name
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    for img_path in file_list:
        img = cv2.imread(str(img_path))
        cv2.imwrite(str(img_dst / img_path.name), apply_clahe(img))
        shutil.copy(labels_dir / f"{img_path.stem}.txt", lbl_dst / f"{img_path.stem}.txt")

# 5. YAML ve Eğitim (5 Sınıf: 4 Anomali + Healthy)
yaml_path = BASE_DIR / "disease_exp03_healthy.yaml"
dataset_cfg = {
    "path": str(BASE_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": class_names
}
with open(yaml_path, "w") as f:
    yaml.dump(dataset_cfg, f, default_flow_style=False, sort_keys=False)

model = YOLO("yolov8s.pt")
results = model.train(
    data=str(yaml_path),
    epochs=60,
    patience=20,
    imgsz=1024,
    batch=8,
    cos_lr=True,
    fliplr=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.2,
    device=0,
    project="/content/runs_q3",
    name="exp03_healthy_clahe_1024"
)

# 6. Test Değerlendirmesi (Hem 5 Sınıf Hem Sadece 4 Anomali Kıyaslaması)
best_weights = Path(results.save_dir) / "weights" / "best.pt"
best_model = YOLO(str(best_weights))
metrics = best_model.val(data=str(yaml_path), split="test", imgsz=1024, project="/content/runs_q3", name="test_eval_exp03", plots=True)

all_ap = metrics.box.all_ap  # shape: [5, 10]
anomaly_ap50 = float(all_ap[:4, 0].mean())
anomaly_ap70 = float(all_ap[:4, 4].mean())
anomaly_ap50_95 = float(all_ap[:4, :].mean())

print("\n============================================================")
print("   GÖREV 3 DENEY 03 SONUÇLARI (HEALTHY SINIFI EKLENMİŞ)")
print("============================================================")
print(f"5 Sınıf Genel mAP50:          {metrics.box.map50:.4f}")
print(f"4 Anomali Sınıfı mAP50:       {anomaly_ap50:.4f}  (Deney 01: 0.4392 | Deney 02: 0.4484)")
print(f"4 Anomali Sınıfı mAP70:       {anomaly_ap70:.4f}  (Deney 01: 0.3651 | Deney 02: 0.3755)")
print(f"4 Anomali Sınıfı mAP50-95:    {anomaly_ap50_95:.4f}  (Deney 01: 0.2747 | Deney 02: 0.2830)")

print("\n--- SINIF BAZLI PERFORMANS ---")
for idx, name in class_names.items():
    p, r, ap50, ap50_95 = metrics.box.class_result(idx)
    ap70 = float(all_ap[idx, 4])
    print(f"{name:<20} | P: {p:.3f} | R: {r:.3f} | AP50: {ap50:.3f} | AP70: {ap70:.3f} | AP50-95: {ap50_95:.3f}")

# 7. Drive Yedekleme
backup_dir = Path("/content/drive/MyDrive/dentex_q3_outputs/exp03_healthy_class")
backup_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(best_weights, backup_dir / "best.pt")
for plot_name in ["confusion_matrix.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "PR_curve.png"]:
    p_file = Path(metrics.save_dir) / plot_name
    if p_file.exists():
        shutil.copy(p_file, backup_dir / plot_name)

print(f"\nModel ve grafikler yedeklendi: {backup_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Görev 2 ağırlık dosyası hazır: /content/gorev2_best.pt
5 Sınıflı Eğitim Etiketleri: {0: 'Impacted', 1: 'Caries', 2: 'Periapical Lesion', 3: 'Deep Caries', 4: 'Healthy'}
Tüm röntgenlerde sağlıklı dişler pseudo-labeling ile işaretleniyor...


100%|██████████| 705/705 [01:36<00:00,  7.31it/s]


Etiketleme Bitti -> Anomali Kutusu: 3529 | Eklenen Sağlıklı Diş Kutusu: 16831
Ultralytics 8.4.220 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dentex_q3_exp03/quadrant-enumeration-disease/disease_exp03_healthy.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, imgsz=1024, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, 